# Conservative 2D regrid — regions (grid → country/state polygons)

Gridded data → arbitrary polygon regions (countries, states, watersheds,
ocean basins) is a canonical xagg-style workflow. Because regions aren't
a grid at all, `.conservative` can't express it;
`ConservativeRegridder.from_polygons` takes a numpy array of shapely
polygons and produces one conservatively-averaged value per region.

This notebook uses **xarray's air-temperature tutorial dataset** (NMC
reanalysis, North America, 2.5° × 2.5°) and aggregates it onto **US
states** built from the `geoda.ncovr` county boundaries shipped in
`geodatasets`.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import geopandas as gpd
import geodatasets

import xarray_regrid  # noqa: F401
from xarray_regrid import ConservativeRegridder, polygons_from_coords

## Source — NCEP reanalysis surface air temperature

The tutorial dataset is 4×daily surface air temperature on a 2.5°
lat/lon grid covering North America. We take the annual mean over
2013–2014 and convert to °C. Longitudes are shifted from the native
[0, 360) convention to [-180, 180) and latitudes flipped to ascending
so the grid aligns with the state polygons' CRS.

In [ ]:
ds = xr.tutorial.open_dataset("air_temperature")
air = (ds["air"].mean("time") - 273.15).sortby("lat")
air = air.assign_coords(lon=(((air.lon + 180) % 360) - 180)).sortby("lon")
air.attrs["units"] = "degC"
air.name = "mean_air_temperature"
air

## Regions — US states from `geodatasets`

`geodatasets.get_path("geoda.ncovr")` returns a GeoPackage of the 49
contiguous-US counties (+ DC). Dissolving on `STATE_NAME` gives one
(Multi)Polygon per state — exactly the input
`ConservativeRegridder.from_polygons` expects. Note the air-temperature
grid does not cover Alaska or Hawaii, which is also why ncovr is a
natural fit.

In [ ]:
counties = gpd.read_file(geodatasets.get_path("geoda.ncovr"))
states = (
    counties.dissolve(by="STATE_NAME", aggfunc="first")
    .reset_index()[["STATE_NAME", "geometry"]]
    .sort_values("STATE_NAME")
    .reset_index(drop=True)
)
print(f"{len(states)} states/DC, CRS={states.crs}, "
      f"bounds={states.total_bounds.round(1).tolist()}")
states.head(3)

## Build the regridder and apply

`from_polygons` takes source + target polygon arrays. Source polygons
are built from the 1D grid coords via `polygons_from_coords`; target
polygons are the states' `geometry` column as a numpy array. The data
is flattened to a single `src_cell` dimension to match.

In [ ]:
src_polys = polygons_from_coords(air.lon.values, air.lat.values)
tgt_polys = states.geometry.to_numpy()

rgr = ConservativeRegridder.from_polygons(
    source_polygons=src_polys,
    target_polygons=tgt_polys,
    source_dim="src_cell",
    target_dim="state",
    target_coords=xr.Dataset(coords={"state": states.STATE_NAME.values}),
)

src_flat = xr.DataArray(air.values.ravel(), dims=("src_cell",))
state_mean = rgr.regrid(src_flat)
state_mean.attrs["units"] = "degC"
state_mean.to_series().sort_values().round(2)

## Map + ranked bar chart

States filled by area-weighted mean temperature, with the source grid
shown underneath for reference.

In [ ]:
fig, (ax_map, ax_bar) = plt.subplots(
    1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [1.6, 1]},
)

air.plot(ax=ax_map, cmap="coolwarm", alpha=0.5,
         cbar_kwargs={"shrink": 0.65, "label": "grid mean T [°C]"})

states_plot = states.assign(mean_T=state_mean.values)
states_plot.plot(
    column="mean_T", cmap="coolwarm", ax=ax_map,
    edgecolor="black", linewidth=0.4,
    vmin=float(air.min()), vmax=float(air.max()),
)
ax_map.set_xlim(-128, -65); ax_map.set_ylim(22, 52)
ax_map.set_title("Annual mean surface T — states vs. source grid")
ax_map.set_xlabel("longitude"); ax_map.set_ylabel("latitude")

ordered = state_mean.to_series().sort_values()
colors = plt.cm.coolwarm(
    (ordered.values - ordered.min()) / (ordered.max() - ordered.min())
)
ax_bar.barh(ordered.index, ordered.values, color=colors)
ax_bar.set_xlabel("area-weighted mean T [°C]")
ax_bar.tick_params(axis="y", labelsize=7)
ax_bar.grid(axis="x", alpha=0.3)
plt.tight_layout()

## Conservation check

The area-weighted sum over regions (using the regridder's internal
area matrix) should match the direct A·s computation — this is the
conservation property the method is named for.

In [ ]:
A = rgr._areas                                # sparse (n_states, n_src)
tgt_area = np.ravel(A.sum(axis=1).todense())
src_cover = np.ravel(A.sum(axis=0).todense())

direct = float((air.values.ravel() * src_cover).sum())
via_regrid = float((state_mean.values * tgt_area).sum())
print(f"direct   A·s            : {direct:.6f}")
print(f"Σ state_mean · a_state  : {via_regrid:.6f}")
print(f"relative error          : {abs(direct - via_regrid) / abs(direct):.2e}")

## Reuse: persist the regridder, apply to summer vs. winter

The weight matrix is reusable across any source field on the same
grid. Save once, reload to skip the intersection build, then apply to
seasonally-filtered data to get per-state JJA vs. DJF means.

In [ ]:
import tempfile
from pathlib import Path
path = Path(tempfile.gettempdir()) / "states_regridder.nc"
rgr.to_netcdf(path)
print(f"wrote {path.name}  ({path.stat().st_size / 1024:.1f} KB)")

rgr2 = ConservativeRegridder.from_netcdf(path)

def seasonal_mean(months):
    sub = ds["air"].sel(time=ds["time.month"].isin(months)).mean("time") - 273.15
    sub = sub.sortby("lat")
    sub = sub.assign_coords(lon=(((sub.lon + 180) % 360) - 180)).sortby("lon")
    flat = xr.DataArray(sub.values.ravel(), dims=("src_cell",))
    return rgr2.regrid(flat)

summer = seasonal_mean([6, 7, 8])
winter = seasonal_mean([12, 1, 2])
amplitude = (summer - winter).to_series().rename("JJA − DJF [°C]").round(1)
print("largest seasonal swing:")
print(amplitude.sort_values(ascending=False).head(5))
print("\nsmallest seasonal swing (maritime / subtropical):")
print(amplitude.sort_values().head(5))